# 05 maps: the current state with the assignments

Draws the tables `04_assigned_coverage.sql` builds into two PDFs on the stage `@td_maps`, inside Snowflake.
Nothing here changes a territory: it only draws `TD_MAP_PIECE`, `TD_MAP_CELL`, `TD_MAP_ZIP` and `TD_MAP_WHOLESALER`.

**Before the first run:** a warehouse-runtime notebook in `ssdb_integration.lr_strategy`, with the packages
`geopandas` and `matplotlib` added (Packages menu, Anaconda channel). Run `03_assignments.sql`, then `04_assigned_coverage.sql`.

**Run:** Run all.  The PDFs land in `@td_maps/<run time>/`: Catalog, Database Explorer, `LR_STRATEGY`, Stages, `TD_MAPS`, then download.

- `td_maps_overview.pdf`: per role, the kept holders on each ZIP (overlap); per estimate, where opportunity is open or has no
  holder; per estimate, the main map's opportunity by ZIP; per firm, main-map sales by ZIP.
- `td_maps_wholesalers.pdf`: one page per kept wholesaler with a role: their pieces in role (sole or shared with another kept
  holder of the role) and out of role, their assigned states outlined, and their numbers.

The PDFs hold wholesaler names and sales: they stay with the owner.  Alaska, Hawaii and the territories are left out.
ZIPs with no ZCTA shape (about 1.3% of the opportunity) are in the numbers but cannot be drawn.

In [ ]:
# Setup: the only cell to edit
from snowflake.snowpark.context import get_active_session

session = get_active_session()
DB = "ssdb_integration.lr_strategy"
UNION_TBL = f"{DB}.sales_opp_2026_11_9"
ZCTA_TBL = f"{DB}.census_zip_codes_zcta"
STATE_TBL = f"{DB}.census_states"
STAGE = f"@{DB}.td_maps"
SIMPLIFY_M = 100   # ZIP outlines simplified to 100 m: enough for a page, much faster
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.td_maps DIRECTORY = (ENABLE = TRUE)").collect()

In [ ]:
# Read the tables (no territory is changed: these are the 04 tables as they are)
import geopandas as gpd
import pandas as pd
from shapely import wkb

NOT_CONUS = ("AK", "HI", "PR", "GU", "VI", "AS", "MP")
NOT_CONUS_FIPS = ("02", "15", "60", "66", "69", "72", "78")


def q(sql):
    return session.sql(sql).to_pandas()


def shapes(sql):
    d = q(sql)
    g = gpd.GeoDataFrame(d.drop(columns="WKB"), geometry=[wkb.loads(bytes(b)) for b in d["WKB"]], crs="EPSG:4326")
    return g.to_crs(epsg=5070)


nc = ", ".join(f"'{s}'" for s in NOT_CONUS)
zips = shapes(f"""
    SELECT z.geo_id AS zip_code, ST_ASWKB(ST_SIMPLIFY(z.geo_boundary, {SIMPLIFY_M})) AS wkb
    FROM {ZCTA_TBL} z
    JOIN (SELECT zip_code FROM {DB}.td_map_cell WHERE COALESCE(state, '') NOT IN ({nc})
          UNION SELECT zip_code FROM {DB}.td_map_piece WHERE COALESCE(state, '') NOT IN ({nc})) k ON k.zip_code = z.geo_id""")
states = shapes(f"""
    SELECT geo_id AS fips, geo_name AS name, ST_ASWKB(ST_SIMPLIFY(geo_boundary, {SIMPLIFY_M * 10})) AS wkb
    FROM {STATE_TBL} WHERE geo_id NOT IN ({", ".join(f"'{s}'" for s in NOT_CONUS_FIPS)})""")
zip_role = q(f"SELECT zip_code, role, n_role_holders, n_role_firms FROM {DB}.td_map_zip")
zip_cell = q(f"""
    SELECT est_source, zip_code, SUM(mm_opp) AS opp,
           SUM(CASE WHEN holder_status = 'KEPT IN ROLE' THEN mm_opp ELSE 0 END) AS opp_in_role,
           SUM(CASE WHEN holder_status = 'KEPT OUT OF ROLE' THEN mm_opp ELSE 0 END) AS opp_out_of_role,
           SUM(CASE WHEN holder_status = 'NOT KEPT (OPEN)' THEN mm_opp ELSE 0 END) AS opp_open,
           SUM(CASE WHEN holder_status = 'NO HOLDER TODAY' THEN mm_opp ELSE 0 END) AS opp_no_holder
    FROM {DB}.td_map_cell GROUP BY est_source, zip_code""")
zip_sales = q(f"""
    SELECT source, LPAD(LEFT(TRIM(zip_code), 5), 5, '0') AS zip_code, SUM(actual_sales_amount) AS sales
    FROM {UNION_TBL}
    WHERE UPPER(TRIM(channel)) IN ('NATIONAL', 'WH', 'FI') AND actual_sales_amount <> 0
    GROUP BY source, LPAD(LEFT(TRIM(zip_code), 5), 5, '0')""")
pieces = q(f"""
    SELECT source, wholesaler, role, zip_code, state, terr_channel, piece_in_role, in_assigned_state, n_role_holders
    FROM {DB}.td_map_piece WHERE kept = 'Y' AND role IN ('NATIONAL', 'WH', 'FI', 'WIFI')""")
wsum = q(f"SELECT * FROM {DB}.td_map_wholesaler WHERE kept = 'Y' AND role IN ('NATIONAL', 'WH', 'FI', 'WIFI')")
print(len(zips), "ZIP shapes,", len(states), "states,", len(pieces), "pieces,", wsum[["SOURCE", "WHOLESALER"]].drop_duplicates().shape[0],
      "kept wholesalers with a role")

In [ ]:
# Drawing: functions of the tables above only
import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

FIPS = {"AL": "01", "AZ": "04", "AR": "05", "CA": "06", "CO": "08", "CT": "09", "DE": "10", "DC": "11", "FL": "12", "GA": "13",
        "ID": "16", "IL": "17", "IN": "18", "IA": "19", "KS": "20", "KY": "21", "LA": "22", "ME": "23", "MD": "24", "MA": "25",
        "MI": "26", "MN": "27", "MS": "28", "MO": "29", "MT": "30", "NE": "31", "NV": "32", "NH": "33", "NJ": "34", "NM": "35",
        "NY": "36", "NC": "37", "ND": "38", "OH": "39", "OK": "40", "OR": "41", "PA": "42", "RI": "44", "SC": "45", "SD": "46",
        "TN": "47", "TX": "48", "UT": "49", "VT": "50", "VA": "51", "WA": "53", "WV": "54", "WI": "55", "WY": "56"}
HOLDER_COLOURS = {"1": "#9ecae1", "2": "#fdae6b", "3": "#e6550d", "4+": "#7f2704"}
STATUS_COLOURS = {"open (holder not kept)": "#d7301f", "no holder today": "#fc8d59", "kept, out of role": "#fdcc8a",
                  "kept, in role": "#e0ecf4"}


def money(x):
    # a bare $ pair starts matplotlib mathtext: escape it
    return f"\\${x / 1e6:,.1f}M" if abs(x) < 1e9 else f"\\${x / 1e9:,.2f}B"


def page(title, note=""):
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_axes([0.02, 0.06, 0.96, 0.86])
    ax.set_axis_off()
    fig.suptitle(title, fontsize=14, x=0.02, ha="left")
    if note:
        fig.text(0.02, 0.02, note, fontsize=7, color="#555")
    return fig, ax


def base(ax, states, bounds=None):
    states.plot(ax=ax, color="#f4f4f4", edgecolor="none")
    if bounds is not None:
        x0, y0, x1, y1 = bounds
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)


def outline(ax, states):
    states.boundary.plot(ax=ax, color="#666", linewidth=0.4)


def legend(ax, colours, counts=None):
    ax.legend(handles=[Patch(facecolor=c, edgecolor="#999", label=f"{k} ({counts.get(k, 0):,})" if counts else k)
                       for k, c in colours.items()], loc="lower left", fontsize=8, frameon=False)


def categorical_page(title, g, column, colours, states, note=""):
    fig, ax = page(title, note)
    base(ax, states)
    counts = g[column].value_counts().to_dict()
    for k, c in colours.items():
        part = g[g[column] == k]
        if len(part):
            part.plot(ax=ax, color=c, edgecolor="none")
    outline(ax, states)
    legend(ax, colours, counts)
    return fig


def quantile_page(title, g, column, cmap, states, note=""):
    fig, ax = page(title, note)
    base(ax, states)
    g = g[g[column] > 0]
    if len(g):
        bins = pd.qcut(g[column].rank(method="first"), q=min(5, len(g)), labels=False)
        cols = plt.get_cmap(cmap)([0.25 + 0.75 * b / max(bins.max(), 1) for b in bins])
        g.plot(ax=ax, color=cols, edgecolor="none")
        edges = [g[column][bins == b] for b in sorted(bins.unique())]
        handles = [Patch(facecolor=plt.get_cmap(cmap)(0.25 + 0.75 * i / max(len(edges) - 1, 1)),
                         label=f"{money(e.min())} to {money(e.max())} ({len(e):,} ZIPs)") for i, e in enumerate(edges)]
        ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=False)
    outline(ax, states)
    return fig


def overview_pages(zips, states, zip_role, zip_cell, zip_sales):
    for role in sorted(zip_role["ROLE"].unique()):
        z = zip_role[zip_role["ROLE"] == role].copy()
        z["band"] = z["N_ROLE_HOLDERS"].map(lambda n: "4+" if n >= 4 else str(int(n)))
        g = zips.merge(z, on="ZIP_CODE")
        two_firms = int((z["N_ROLE_FIRMS"] > 1).sum())
        yield categorical_page(f"{role}: kept holders with an in-role piece on the ZIP", g, "band", HOLDER_COLOURS, states,
                               f"Today's territories as they are. 2 or more is an overlap. ZIPs held by both firms: {two_firms:,}.")
    for est in sorted(zip_cell["EST_SOURCE"].unique()):
        c = zip_cell[zip_cell["EST_SOURCE"] == est].copy()
        c["status"] = "kept, in role"
        for col, label in (("OPP_OUT_OF_ROLE", "kept, out of role"), ("OPP_NO_HOLDER", "no holder today"),
                           ("OPP_OPEN", "open (holder not kept)")):
            c.loc[c[col] > 0, "status"] = label
        tot = c["OPP"].sum()
        note = (f"{est} estimate of the main map, {money(tot)}: open {money(c['OPP_OPEN'].sum())}, no holder "
                f"{money(c['OPP_NO_HOLDER'].sum())}, kept out of role {money(c['OPP_OUT_OF_ROLE'].sum())}. "
                "A ZIP takes the first of open, no holder, out of role that it has any opportunity in.")
        yield categorical_page(f"Main-map opportunity by holder status ({est} estimate)", zips.merge(c, on="ZIP_CODE"), "status",
                               STATUS_COLOURS, states, note)
    for est in sorted(zip_cell["EST_SOURCE"].unique()):
        c = zip_cell[zip_cell["EST_SOURCE"] == est]
        yield quantile_page(f"Main-map opportunity by ZIP ({est} estimate)", zips.merge(c, on="ZIP_CODE"), "OPP", "Blues", states,
                            "Fifths of the ZIPs with opportunity. The two estimates are of one market: never add them.")
    for src in sorted(zip_sales["SOURCE"].unique()):
        s = zip_sales[zip_sales["SOURCE"] == src]
        yield quantile_page(f"Main-map sales by ZIP ({src})", zips.merge(s, on="ZIP_CODE"), "SALES", "Greens", states,
                            "Fifths of the ZIPs with sales. The sale's ZIP as the sales table has it.")


def wholesaler_page(src, name, p, w, zips, states, zip_role):
    role = p["ROLE"].iloc[0]
    g = zips.merge(p, on="ZIP_CODE")
    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_axes([0.02, 0.06, 0.66, 0.86])
    ax.set_axis_off()
    fig.suptitle(f"{name} ({src}), {role}", fontsize=14, x=0.02, ha="left")
    if len(g):
        x0, y0, x1, y1 = g.total_bounds
        pad = max(x1 - x0, y1 - y0) * 0.08 + 20000
        bounds = (x0 - pad, y0 - pad, x1 + pad, y1 + pad)
    else:
        bounds = None
    base(ax, states, bounds)
    near = zips.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]] if bounds else zips.iloc[0:0]
    others = near.merge(zip_role[zip_role["ROLE"] == role], on="ZIP_CODE")
    if len(others):
        others.plot(ax=ax, color="#dddddd", edgecolor="none")
    inr = g[g["PIECE_IN_ROLE"] == 1].drop_duplicates("ZIP_CODE")
    out = g[(g["PIECE_IN_ROLE"] == 0) & ~g["ZIP_CODE"].isin(inr["ZIP_CODE"])].drop_duplicates("ZIP_CODE")
    sole, shared = inr[inr["N_ROLE_HOLDERS"].fillna(0) <= 1], inr[inr["N_ROLE_HOLDERS"].fillna(0) > 1]
    for part, kw in ((sole, dict(color="#2171b5")), (shared, dict(color="#fd8d3c")),
                     (out, dict(facecolor="none", edgecolor="#555", hatch="////", linewidth=0.2))):
        if len(part):
            part.plot(ax=ax, edgecolor=kw.pop("edgecolor", "none"), **kw)
    outline(ax, states)
    codes = str(w["ASSIGNED_STATES"].iloc[0] or "").split()
    assigned = states[states["FIPS"].isin([FIPS.get(c) for c in codes])]
    if len(assigned):
        assigned.boundary.plot(ax=ax, color="black", linewidth=1.6)
    ax.legend(handles=[Patch(facecolor="#2171b5", label=f"in role, sole kept {role} holder ({len(sole):,} ZIPs)"),
                       Patch(facecolor="#fd8d3c", label=f"in role, shared with another kept {role} ({len(shared):,})"),
                       Patch(facecolor="white", edgecolor="#555", hatch="////", label=f"out of role only ({len(out):,})"),
                       Patch(facecolor="#dddddd", label=f"other kept {role} holders' ZIPs"),
                       Patch(facecolor="white", edgecolor="black", linewidth=1.6, label="assigned states")],
              loc="lower left", fontsize=8, frameon=False)
    lines = [f"Role {role}", f"ZIPs held today {int(w['N_ZIPS'].iloc[0]):,}, in role {int(w['N_ZIPS_IN_ROLE'].iloc[0]):,}",
             f"States {w['STATES'].iloc[0] or '-'}", f"Assigned states {' '.join(codes) or '-'}", ""]
    for _, r in w.sort_values("EST_SOURCE").iterrows():
        lines += [f"Opportunity, {r['EST_SOURCE']} estimate", f"  on their pieces  {money(r['OPP_MAIN'])}",
                  f"  in role          {money(r['OPP_IN_ROLE'])}", f"  out of role      {money(r['OPP_OUT_OF_ROLE'])}",
                  f"  shared (overlap) {money(r['OPP_OVERLAP'])}"]
        if pd.notna(r["OPP_IN_ASSIGNED_STATES"]):
            lines.append(f"  in assigned st.  {money(r['OPP_IN_ASSIGNED_STATES'])}")
        if pd.notna(r["SALES_MAIN"]):
            pct = 100 * r["SALES_MAIN_IN_OWN_ZIPS"] / r["SALES_MAIN"] if r["SALES_MAIN"] else 0
            lines += ["", f"Main-map sales {money(r['SALES_MAIN'])}", f"  in their own ZIPs {pct:.0f}%"]
        lines.append("")
    fig.text(0.70, 0.88, "\n".join(lines), fontsize=8.5, va="top", family="monospace")
    return fig


def wholesaler_pages(zips, states, zip_role, pieces, wsum):
    keys = wsum[["SOURCE", "WHOLESALER"]].drop_duplicates().sort_values(["SOURCE", "WHOLESALER"])
    for src, name in keys.itertuples(index=False):
        p = pieces[(pieces["SOURCE"] == src) & (pieces["WHOLESALER"] == name)]
        w = wsum[(wsum["SOURCE"] == src) & (wsum["WHOLESALER"] == name)]
        if len(p):
            yield wholesaler_page(src, name, p, w, zips, states, zip_role)


def write_pdf(path, figs):
    n = 0
    with PdfPages(path) as pdf:
        for fig in figs:
            pdf.savefig(fig)
            plt.close(fig)
            n += 1
    return n

In [ ]:
# Run: the two PDFs, then onto the stage
import datetime
import os
import tempfile

run = datetime.datetime.now().strftime("%Y-%m-%d-%H%M")
out = tempfile.mkdtemp()
for fname, figs in (("td_maps_overview.pdf", overview_pages(zips, states, zip_role, zip_cell, zip_sales)),
                    ("td_maps_wholesalers.pdf", wholesaler_pages(zips, states, zip_role, pieces, wsum))):
    path = os.path.join(out, fname)
    n = write_pdf(path, figs)
    session.file.put(path, f"{STAGE}/{run}/", auto_compress=False, overwrite=True)
    print(f"{STAGE}/{run}/{fname}: {n} pages")
session.sql(f"ALTER STAGE {DB}.td_maps REFRESH").collect()